In [1]:

import pandas as pd
from prophet import Prophet  # Ensure 'prophet' is installed locally
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Load and preprocess the daily dataset
file_path = '/Users/bhavanahiremath/Downloads/SN_d_tot_V2.0.csv'
daily_data = pd.read_csv(file_path, delimiter=';', header=None)

# Assign meaningful column names based on the description in the project PDF
daily_data.columns = ['Year', 'Month', 'Day', 'Date_Fraction', 'Sunspot_Count',
                      'Std_Dev', 'Observations', 'Provisional']

# Create a proper datetime column
daily_data['Date'] = pd.to_datetime(daily_data[['Year', 'Month', 'Day']])

# Select necessary columns for FBProphet and handle missing values (-1)
daily_data = daily_data[['Date', 'Sunspot_Count']]
daily_data = daily_data[daily_data['Sunspot_Count'] != -1]

# Rename columns to match Prophet's requirements
daily_data = daily_data.rename(columns={'Date': 'ds', 'Sunspot_Count': 'y'})

# Initialize the Prophet model
prophet = Prophet()

# Fit the model
prophet.fit(daily_data)

# Make future predictions
future = prophet.make_future_dataframe(periods=365)  # Predicting 365 days into the future
forecast = prophet.predict(future)

# Evaluate the model (using a split for training vs. actual)
split_date = daily_data['ds'].iloc[-365]  # Take the last 365 days as test
train = daily_data[daily_data['ds'] < split_date]
test = daily_data[daily_data['ds'] >= split_date]

# Fit model on training data
prophet.fit(train)
forecast_test = prophet.predict(test[['ds']])

# Evaluation metrics
mae = mean_absolute_error(test['y'], forecast_test['yhat'])
mape = mean_absolute_percentage_error(test['y'], forecast_test['yhat'])
r2 = r2_score(test['y'], forecast_test['yhat'])

# Plot the results
fig = prophet.plot(forecast)
plt.title('Daily Sunspot Forecast (Prophet)')
plt.show()

# Display metrics
print('Evaluation Results:')
print(f'Mean Absolute Error: {mae}')
print(f'Mean Absolute Percentage Error: {mape}')
print(f'R² Score: {r2}')


ModuleNotFoundError: No module named 'prophet'